In [ ]:
import math
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Dict

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv
    from LabUserTileRequest import UserTileRequestEvents
    from LabEnvWrapper import EnvWrapper

import sys
# sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
from Common.Utils import save_training_results


In [ ]:
# --- 1. CONFIGURATION & HYPERPARAMETERS (Section VII-B) ---
class Config:
    n_episodes: int = 300
    n_nodes: int = 3
    n_users: int = 1
    step_size: float = 10.00
    arrival_rate: float = 10.0  # users per second
    alpha: float = 0.5
    n_videos: int = 50
    n_gops: int = 30
    n_layers: int = 2
    n: int = 4
    m: int = 3
    n_tiles: int = n * m
    tiles_per_viewport: int = 4

    base_tile_mb = 2e6 / n_tiles
    enh_tile_mb = 15e6 / n_tiles

    max_capacity: float = 500e6  # 500 MB
    cache_capacity_percent: float = 0.1  # 10% of the total video size
    cache_size: int = int(n_videos * cache_capacity_percent)
    cache_capacity_mb: float = (
        n_gops * n_tiles * base_tile_mb +
        n_gops * tiles_per_viewport * enh_tile_mb
    ) * cache_size

    # Hyperparameters for RL
    epsilon_start: float = 1.0
    epsilon_min: float = 0.005
    epsilon_decay: float = 0.987
    gamma: float = 0.99
    learning_rate: float = 1e-3
    batch_size: int = 32
    capacity: int = 10000
    window_len: int = 3  # LSTM sequence length (history window)

    h_short: int = 300   # sliding windows for popularity (Section VI-A)
    h_long: int = 1000

    r_base: float = 30.0 # PSNR reward for base layer (Section VI-C)
    r_enh: float = 10.0  # PSNR reward for enhancement layer
    penalty: float = 0.0 # fetch penalty (implicit in paper)

    # CPT parameters
    theta: float = 0.5
    lam: float = 3.7183

    @property
    def state_dim(self) -> int:
        # 10*C + 2 = (2C + 2Ck) * 2 + 2 (Section VI-A)
        return 10 * self.cache_size + 2

    @property
    def action_dim(self) -> int:
        # |A| = 5C + 1 (Section VI-B)
        return 5 * self.cache_size + 1

In [ ]:
# --- 3. REPLAY BUFFER ---
class ReplayBuffer:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*batch)
        return (
            np.stack(state),
            np.array(action),
            np.array(reward, dtype=np.float32),
            np.stack(next_state),
            np.array(done, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)

In [ ]:
# --- 2. DEEP Q-NETWORK (Section VI & VII-B) ---
class DQN(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super(DQN, self).__init__()
        # 4 Fully Connected Layers (Input, 2 Hidden, Output)
        # Hidden layers have 5C+1 nodes (Source 469)
        hidden_dim = output_dim 
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        
        return self.fc3(x) # Linear activation for output (Source 470)

In [ ]:
class FeatureAdapter:
    def __init__(self, env_cache: CacheEngineEnv, cfg: Config):
        self.env_cache = env_cache
        self.cfg = cfg

        self.video_hist_short = deque(maxlen=cfg.h_short)
        self.video_hist_long = deque(maxlen=cfg.h_long)
        self.tile_hist_short = deque(maxlen=cfg.h_short)
        self.tile_hist_long = deque(maxlen=cfg.h_long)

        self.video_freq_short = defaultdict(int)
        self.video_freq_long = defaultdict(int)
        self.tile_freq_short = defaultdict(int)
        self.tile_freq_long = defaultdict(int)

        self.video_short_count = defaultdict(int)
        self.video_long_count  = defaultdict(int)


    def reset_history(self):
        queues = (
            self.video_hist_short,
            self.video_hist_long,
            self.tile_hist_short,
            self.tile_hist_long,
        )
        freqs = (
            self.video_freq_short,
            self.video_freq_long,
            self.tile_freq_short,
            self.tile_freq_long,
        )
        counts = (
            self.video_short_count,
            self.video_long_count,
        )
        for q in queues:
            q.clear()
        for f in freqs:
            f.clear()
        for c in counts:
            c.clear()

    def update_history(self, request: Dict):
        vid = request["video"]
        tiles = request["tiles"]

        self._update_window(self.video_hist_short, self.video_freq_short, vid)
        self._update_window(self.video_hist_long, self.video_freq_long, vid)

        for t in tiles:
            self._update_window(self.tile_hist_short, self.tile_freq_short, t)
            self._update_window(self.tile_hist_long, self.tile_freq_long, t)

    @staticmethod
    def _update_window(self, window: deque, freq: Dict, item):
        if len(window) == window.maxlen:
            old_item = window.popleft()
            freq[old_item] -= 1
            if freq[old_item] == 0:
                del freq[old_item]
        window.append(item)
        freq[item] += 1

    def build_state(self, candidate_request: Dict) -> np.ndarray:
        cache_entries = self.env_cache.get_cache_entities()
        C = self.env_cache.get_cache_capacity()
        K = self.env_cache.get_viewport_tile_budget()
        xs, xl, ys, yl = [], [], [], []

        for video_id, meta in cache_entries[:C]:
            xs.append(self.video_freq_short.get(video_id, 0))
            xl.append(self.video_freq_long.get(video_id, 0))
            viewport_tiles = meta.get("viewport_tiles", [])[:K]

            for tile_id in viewport_tiles:
                ys.append(self.tile_freq_short.get(tile_id, 0))
                yl.append(self.tile_freq_long.get(tile_id, 0))

        # Pad to full dimensionality
        while len(xs) < C:
            xs.append(0)
            xl.append(0)
        while len(ys) < C * K:
            ys.append(0)
            yl.append(0)

        vid = candidate_request["video"]
        Zs = self.video_freq_short.get(vid, 0)
        Zl = self.video_freq_long.get(vid, 0)

        state = np.array(xs + xl + ys + yl + [Zs, Zl], dtype=np.float32)
        assert state.shape[0] == self.cfg.state_dim, "State dimension mismatch"
        return state

    def build_state_2(
        self,
        adapter,
        user
    ):
        """
        Builds S = (X_s, X_l, Y_s, Y_l, Z_s, Z_l)
        """
        cur_video_id = user['video']

        C = self.env_cache.unit_mapper.max_units
        K = self.env_cache.unit_mapper.viewport_tiles

        # ---- Order cached videos (LRU order) ----
        cached_videos = self.env_cache.get_cache_entities()
        Xs, Xl = [], []
        Ys, Yl = [], []

        for key in cached_videos:
            video_id, layer_id, tile_id, gop_id = key

            # ---- X features ----
            Xs.append(self.video_short_count.get(video_id, 0))
            Xl.append(self.video_long_count.get(video_id, 0))

            # ---- Y features ----
        viewport_tiles = user['viewport_tiles'][:K]
        for tile_id in viewport_tiles[:K]:
            Ys.append(self.tile_short_count.get(tile_id, 0))
            Yl.append(self.tile_long_count.get(tile_id, 0))

        # ---- Padding (important) ----
        while len(Xs) < C:
            Xs.append(0)
            Xl.append(0)

        while len(Ys) < C * K:
            Ys.append(0)
            Yl.append(0)

        # ---- Z features (candidate) ----
        Zs = adapter.video_short_count.get(cur_video_id, 0)
        Zl = adapter.video_long_count.get(cur_video_id, 0)

        state = np.array(
            Xs + Xl + Ys + Yl + [Zs, Zl],
            dtype=np.float32
        )

        return state

    def build_state_3(self, candidate_request: Dict) -> np.ndarray:
        cfg = self.cfg
        env = self.env_cache

        state = np.zeros(cfg.state_dim, dtype=np.float32)
        offset = 0

        # 2C: cache indicator for base and enhancement layers
        for i in range(cfg.cache_size):
            state[offset + i] = 1.0 if env.is_cached(i, layer=0) else 0.0
        offset += cfg.cache_size
        for i in range(cfg.cache_size):
            state[offset + i] = 1.0 if env.is_cached(i, layer=1) else 0.0
        offset += cfg.cache_size

    #     # 2Ck: short-term and long-term video popularities
    #     for i in range(cfg.cache_size):
    #         state[offset + i] = (
    #             self.video_freq_short.get(i, 0) / cfg.h_short
    #         )
    #     offset += cfg.cache_size
    #     for i in range(cfg.cache_size):
    #         state[offset + i] = (
    #             self.video_freq_long.get(i, 0) / cfg.h_long
    #         )
    #     offset += cfg.cache_size

    #     # 2Ck: short-term and long-term tile popularities
    #     for i in range(cfg.cache_size):
    #         state[offset + i] = (
    #             self.tile_freq_short.get(i, 0) / cfg.h_short
    #         )
    #     offset += cfg.cache_size
    #     for i in range(cfg.cache_size):
    #         state[offset + i] = (
    #             self.tile_freq_long.get(i, 0) / cfg.h_long
    #         )
    #     offset += cfg.cache_size

    #     # 2: size of requested base and enhancement layers
    #     req_video = candidate_request["video"]
    #     state[offset] = env.video_repo.get_layer_size(
    #         req_video, layer=0
    #     ) / env.max_capacity
    #     offset += 1
    #     state[offset] = env.video_repo.get_layer_size(
    #         req_video, layer=1
    #     ) / env.max_capacity

    #     return state

In [ ]:
class CachingAgent:
    def __init__(self, env: CacheEngineEnv, cfg: Config):
        self.env = env
        self.cfg = cfg
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.policy_net = DQN(cfg.state_dim, cfg.action_dim).to(self.device)
        self.target_net = DQN(cfg.state_dim, cfg.action_dim).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=cfg.lr)
        self.memory = ReplayBuffer(cfg.buffer_size)

        self.feature_adapter = FeatureAdapter(env, cfg)

        # per-episode bookkeeping
        self.global_step = 0

    def select_action(self, state: np.ndarray) -> int:
        """
        ε-greedy using the fixed ε from Table II (Section VI-D).
        """
        if random.random() < self.cfg.epsilon:
            return random.randint(0, self.cfg.action_dim - 1)
        state_t = torch.from_numpy(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.policy_net(state_t)
        return int(q_values.argmax(dim=1).item())

    def optimize(self):
        if len(self.memory) < self.cfg.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.memory.sample(
            self.cfg.batch_size
        )

        states_t = torch.from_numpy(states).float().to(self.device)
        actions_t = torch.from_numpy(actions).long().unsqueeze(1).to(self.device)
        rewards_t = torch.from_numpy(rewards).unsqueeze(1).to(self.device)
        next_states_t = torch.from_numpy(next_states).float().to(self.device)
        dones_t = torch.from_numpy(dones).unsqueeze(1).to(self.device)

        q_vals = self.policy_net(states_t).gather(1, actions_t)
        with torch.no_grad():
            max_next_q = self.target_net(next_states_t).max(dim=1, keepdim=True)[0]
            targets = rewards_t + self.cfg.gamma * max_next_q * (1.0 - dones_t)

        loss = nn.MSELoss()(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10.0)
        self.optimizer.step()

        self.global_step += 1
        if self.global_step % self.cfg.target_update_freq == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        return float(loss.item())

    # --------------------------------------------------------------------- #
    # Training Loop (Episodes correspond to streaming sessions)
    # --------------------------------------------------------------------- #
    def train(self, num_episodes: int | None = None):
        num_episodes = num_episodes or self.cfg.train_epochs
        stats = {"reward": [], "loss": []}

        for episode in range(num_episodes):
            obs, info = self.env.reset()
            self.feature_adapter.reset_history()
            ep_reward = 0.0
            losses = []

            done = False
            while not done:
                requests = [
                    req for req in info["users_requests"]
                    if req["gop"] >= 0  # keep active users only
                ]
                actions = []

                for req in requests:
                    self.feature_adapter.update_history(req)
                    state = self.feature_adapter.build_state(req)
                    action = self.select_action(state)
                    actions.append(action)

                next_obs, reward, done, info = self.env.step(actions)
                ep_reward += reward

                # Construct next-state for each active request (Section VI)
                # The paper uses a single aggregated reward, which we replay once
                if requests:
                    next_states = []
                    for req in info["users_requests"]:
                        next_state = self.feature_adapter.build_state(req)
                        next_states.append(next_state)

                    # Here we only store the first (representative) transition,
                    # because the environment aggregates reward per-step.
                    s = self.feature_adapter.build_state(requests[0])
                    ns = next_states[0]
                    self.memory.push(s, actions[0], reward, ns, float(done))

                    loss = self.optimize()
                    if loss is not None:
                        losses.append(loss)

            stats["reward"].append(ep_reward)
            stats["loss"].append(np.mean(losses) if losses else np.nan)
            print(f"[Episode {episode+1:03d}] reward={ep_reward:.2f}")

        return stats

    def evaluate(self, num_episodes: int = 5):
        self.policy_net.eval()
        returns = []
        with torch.no_grad():
            for _ in range(num_episodes):
                obs, info = self.env.reset()
                self.feature_adapter.reset_history()
                done = False
                total_r = 0.0
                while not done:
                    actions = []
                    for req in info["users_requests"]:
                        self.feature_adapter.update_history(req)
                        state = self.feature_adapter.build_state(req)
                        action = self.select_action(state)
                        actions.append(action)
                    obs, reward, done, info = self.env.step(actions)
                    total_r += reward
                returns.append(total_r)
        self.policy_net.train()
        return returns


In [ ]:
# --- 4. DRL AGENT ---
class DRLAgent:
    def __init__(self, config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Evaluation and Target Networks
        self.policy_net = DQN(config.STATE_DIM, config.ACTION_DIM).to(self.device)
        self.target_net = DQN(config.STATE_DIM, config.ACTION_DIM).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=config.LR)
        self.memory = ReplayBuffer(config.BUFFER_SIZE)
        self.steps_done = 0
        
    def select_action(self, state):
        # Epsilon-Greedy Policy (Source 313)
        if random.random() < self.config.EPSILON:
            return random.randint(0, self.config.ACTION_DIM - 1)
        else:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                q_values = self.policy_net(state_t)
                return q_values.argmax().item()

    def train(self):
        if len(self.memory) < self.config.BATCH_SIZE:
            return None # Not enough samples yet

        # Sample mini-batch (Source 328)
        states, actions, rewards, next_states, dones = self.memory.sample(self.config.BATCH_SIZE)

        state_batch = torch.FloatTensor(np.array(states)).to(self.device)
        action_batch = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        reward_batch = torch.FloatTensor(rewards).unsqueeze(1).to(self.device)
        next_state_batch = torch.FloatTensor(np.array(next_states)).to(self.device)
        done_batch = torch.FloatTensor(dones).unsqueeze(1).to(self.device)

        # Compute Q(s, a)
        curr_q_values = self.policy_net(state_batch).gather(1, action_batch)

        # Compute Max Q(s', a') from Target Net (Fixed Target Mechanism)
        next_q_values = self.target_net(next_state_batch).max(1)[0].unsqueeze(1)
        expected_q_values = reward_batch + (self.config.GAMMA * next_q_values * (1 - done_batch))

        # Loss Function (MSE) (Source 349)
        loss = nn.MSELoss()(curr_q_values, expected_q_values)

        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Update Target Network periodically (Source 335)
        self.steps_done += 1
        if self.steps_done % self.config.TARGET_UPDATE_FREQ == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        
        return loss.item()

In [ ]:
class AgentAdapter:
    """
    Wraps the provided LabEnv to make it compatible with the DQN Agent.
    Handles:
    1. Feature Extraction (converting raw Env observations to State Vectors).
    2. Action Decoding (converting DQN integer actions to Env commands).
    3. History Tracking (Short vs Long term memory).
    """
    def __init__(self, lab_env, config):
        self.env = lab_env
        self.c = config

        # History Tracking (Required for State Features x, y, z)
        # We need to track these externally if the LabEnv doesn't provide processed features
        self.video_history_short = deque(maxlen=self.c.H_SHORT)
        self.video_history_long = deque(maxlen=self.c.H_LONG)
        self.tile_history_short = deque(maxlen=self.c.H_SHORT)
        self.tile_history_long = deque(maxlen=self.c.H_LONG)

        self.video_short_count = defaultdict(int)
        self.video_long_count  = defaultdict(int)


    def update_history(self, request):
        """
        Updates the history queues based on the new request from LabEnv.
        current_request: Expected to contain {video_id, tile_ids}
        """
        vid_id = request['video']
        gop_id = request['gop']
        tile_ids = request['tiles']

        # ---- SHORT window ----
        if len(self.video_history_short) == self.video_history_short.maxlen:
            old = self.video_history_short.popleft()
            self.video_short_count[old] -= 1
            if self.video_short_count[old] == 0:
                del self.video_short_count[old]

        self.video_history_short.append(vid_id)
        self.video_short_count[vid_id] += 1

        # ---- LONG window ----
        if len(self.video_history_long) == self.video_history_long.maxlen:
            old = self.video_history_long.popleft()
            self.video_long_count[old] -= 1
            if self.video_long_count[old] == 0:
                del self.video_long_count[old]

        self.video_history_long.append(vid_id)
        self.video_long_count[vid_id] += 1
        
    def get_state_vector(self, candidate_id, is_tile_candidate, tile_id=None):
        """
        Constructs the neural network input vector (10C + 2) from the LabCacheEngine status
        and our local history queues.
        """
        state = []

        # Access the current cache state from your Engine
        # Assuming env.cache_engine.get_slots() returns list of cached Video IDs
        current_cache_slots = self.env.mec_cache.get_slots() 
        current_cache_tiles = self.env.mec_cache.get_cached_tiles() # Should return list of lists

        print("Current Cache Slots:", current_cache_slots)
        print("Current Cache Tiles:", current_cache_tiles)

        return np.array(state, dtype=np.float32)

    def reset(self):
        _, info = self.env.reset()
        
        # Clear history queues
        self.video_history_short.clear()
        self.video_history_long.clear()
        self.tile_history_short.clear()
        self.tile_history_long.clear()
        
        # Initial State Vector
        state = self.get_state_vector(candidate_id=None, is_tile_candidate=False)
        
        return state, info

In [ ]:
from functools import cache


if __name__ == "__main__":
    print("--- Starting DRL Caching System ---")

    # 1. Load Configuration
    cfg = Config()
    
    # 2. Initialize Environment
    du_caches = []

    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
        cache_capacity=cfg.cache_capacity_mb,
        # policy=SvcLruPolicy(max_size=max_capacity)
    )

    # Initialize User Environment
    users_env = UserTileRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        users_viewport_tiles=None,
        requested_videos=None,
        users_arrivals=None,
        arrival_rate=cfg.arrival_rate,
        alpha=cfg.alpha
    )
    
    P = cfg.n_nodes; max_U = cfg.n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,  # 640 Mbps -> 80e6 B/s
        R_C_M=125e6, # 1 Gbps -> 125e6 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),    # 160 Mbps -> 20e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,  # 1 ms
        mec_fixed_delay=0.005, # 5 ms
        cloud_fixed_delay=0.1  # 100 ms
    )

    env_cache = EnvWrapper(
        n=cfg.n,
        n_layers=cfg.n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        theta=cfg.theta,
        lam=cfg.lam
    )
    
    # 3. Initialize Agent Adapter
    adapter = AgentAdapter(env_cache, cfg)
    
    # 4. Initialize DRL Agent
    agent = DRLAgent(cfg)
    
    ### 5. Training Loop ###
    obs, info = adapter.reset()
    done = False
    episode_reward = 0

    for step in count():
        
        reqs_state = info['users_requests']
        active_users = [
            req  for req in reqs_state if req['gop'] < cfg.n_gops
        ]

        actions = []
        for user in active_users:
            print("Processing User Request:", user)

            video = user['video']
            tiles = user['tiles']
            gop = user['gop']

            adapter.update_history(user)

            state = build_state(
                adapter,
                env_cache.mec_cache,
                user
            )

            action = agent.select_action(state)

        obs, rewards, done, info = env_cache.step(actions)

        if done:
            break

        print(f"Step {step}, Active Users: {len(active_users)}")
        # print(f"Request State: {reqs_state}")
        # print(f"Action: {actions}")
        # print(f"Next Request State: {reqs_next_state}")
        # print(
        #     f"Reward: {rewards}, "
        #     f"Cache Hits: {enhanced_layer_cache_hits + base_layer_cache_hits}, "
        #     f"Cache Misses: {enhanced_layer_cache_misses + base_layer_cache_misses}"
        # )
        print("-----")
    print("--- Training Completed ---")


--- Starting DRL Caching System ---
Current Cache Slots: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
Current Cache Tiles: [[11, 8, 6, 0, 7, 10, 2, 9], [6, 8, 1, 2, 4, 3], [4, 9, 5, 1, 6], [10, 3, 1, 8, 0, 11, 6], [5, 7, 8, 1, 0], [7, 6, 11, 5, 4, 8, 3, 2], [5, 9, 8, 0, 3, 7, 11], [2, 5, 4, 11, 8, 9, 7], [11, 4, 8, 10, 2, 6, 9], [5, 0, 11, 9, 7, 6], [7, 5, 2, 10, 8, 0], [8, 11, 10, 9, 5, 2, 0], [6, 5, 11, 7, 8, 10, 1, 4], [1, 2, 9, 8, 4, 5], [7, 6, 1, 0, 3, 11], [2, 8, 9, 7, 1, 4, 10], [5, 11, 6, 9, 8, 10], [3, 1, 7, 0, 11, 8, 9, 6], [7, 1, 10, 9, 4, 5], [5, 10, 9, 11, 3, 7, 4, 8, 6, 2], [3, 7, 6, 10, 9, 4, 8, 1, 5, 2], [9, 6, 7, 10, 0, 11, 1], [5, 6, 9, 2, 7, 0, 10, 8], [1, 3, 4, 0, 10, 2], [0, 7, 9, 8, 6, 5, 4, 10], [9, 6, 0, 8, 10, 4], [11, 3, 7, 10, 1], [10, 7, 6, 5, 9, 1, 4, 2, 0], [1, 7, 6, 10, 4, 11, 5, 2, 3, 0], [7, 0, 8, 1, 5, 4, 10,

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x22 and 52x26)